In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col, lit
from datetime import datetime, timedelta
import os, shutil, time, json
import pandas as pd


spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara-Streaming") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

**A. Menyiapkan Streaming Source**

In [6]:
skema_pesanan = StructType([
    StructField("pesanan_id",   StringType(),    True),
    StructField("nama_cabang",  StringType(),    True),
    StructField("nama_menu",    StringType(),    True),
    StructField("qty",          IntegerType(),   True),
    StructField("harga_satuan", IntegerType(),   True),
    StructField("waktu_pesan",  TimestampType(), True),
])

folder_stream = "tugas10_stream"
folder_tmp    = "tugas10_tmp"     # tempat menulis berkas sebelum dipindah ke folder stream

# Bersihkan sisa eksekusi sebelumnya
for f in [folder_stream, folder_tmp, "tugas10_output", "tugas10_checkpoint"]:
    shutil.rmtree(f, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)
os.makedirs(folder_tmp, exist_ok=True)

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("df_stream.isStreaming:", df_stream.isStreaming)
df_stream.printSchema()

df_stream.isStreaming: True
root
 |-- pesanan_id: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- waktu_pesan: timestamp (nullable = true)



**B. Agregasi per Cabang**

In [13]:
df_stream = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_cabang_df = df_stream.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_cabang = agregasi_cabang_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi_cabang aktif:", query_cabang.isActive)

Query agregasi_cabang aktif: True


**C. Agregasi Berbasis Window Waktu**

In [14]:
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_waktu_df = df_stream_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_waktu = agregasi_waktu_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query agregasi_waktu aktif:", query_waktu.isActive)


Query agregasi_waktu aktif: True


**D. Menguji dengan Data Simulasi**

In [26]:
import os
import time

folder_streaming = "tugas10_stream"
os.makedirs(folder_streaming, exist_ok=True)

# BATCH 1 
data_batch1 = (
    "pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan\n"
    "P001,Cabang Pusat,Kopi Susu,2,18000,2026-11-10 10:00:00\n"
    "P002,Cabang Utara,Espresso,1,20000,2026-11-10 10:01:00\n"
)
with open(f"{folder_streaming}/batch1.csv", "w") as f:
    f.write(data_batch1)

print("Batch 1 dikirim ke folder streaming...")
time.sleep(6) # Tunggu proses streaming

print("\n [HASIL] Agregasi Cabang - Setelah Batch 1")
spark.sql("SELECT * FROM agregasi_cabang").show(truncate=False)

Batch 1 dikirim ke folder streaming...

 [HASIL] Agregasi Cabang - Setelah Batch 1
+--------------+----------------+--------------+
|nama_cabang   |total_pendapatan|jumlah_pesanan|
+--------------+----------------+--------------+
|Semarang      |110000          |1             |
|Cabang Selatan|44000           |1             |
|Cabang Utara  |45000           |2             |
|Solo          |70000           |2             |
|Yogyakarta    |72000           |1             |
|Magelang      |131000          |3             |
|Cabang Pusat  |99000           |3             |
+--------------+----------------+--------------+



In [27]:
data_batch2 = (
    "pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan\n"
    "P003,Cabang Pusat,Americano,3,15000,2026-11-10 10:02:30\n"
    "P004,Cabang Selatan,Latte,2,22000,2026-11-10 10:03:00\n"
)
with open(f"{folder_streaming}/batch2.csv", "w") as f:
    f.write(data_batch2)

print("Batch 2 dikirim ke folder streaming...")
time.sleep(6)

print("\n[HASIL] Agregasi Cabang - Setelah Batch 2")
spark.sql("SELECT * FROM agregasi_cabang").show(truncate=False)

Batch 2 dikirim ke folder streaming...

[HASIL] Agregasi Cabang - Setelah Batch 2
+--------------+----------------+--------------+
|nama_cabang   |total_pendapatan|jumlah_pesanan|
+--------------+----------------+--------------+
|Semarang      |110000          |1             |
|Cabang Selatan|44000           |1             |
|Cabang Utara  |45000           |2             |
|Solo          |70000           |2             |
|Yogyakarta    |72000           |1             |
|Magelang      |131000          |3             |
|Cabang Pusat  |99000           |3             |
+--------------+----------------+--------------+



In [28]:
data_batch3 = (
    "pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan\n"
    "P005,Cabang Utara,Manual Brew,1,25000,2026-11-10 10:04:15\n"
    "P006,Cabang Pusat,Kopi Susu,1,18000,2026-11-10 10:05:00\n"
)
with open(f"{folder_streaming}/batch3.csv", "w") as f:
    f.write(data_batch3)

print(">>> Batch 3 dikirim ke folder streaming...")
time.sleep(6)

print("\n--- [HASIL] Agregasi Cabang - Setelah Batch 3 (Terakumulasi) ---")
spark.sql("SELECT * FROM agregasi_cabang").show(truncate=False)

print("\n[HASIL] Agregasi Berbasis Window Waktu (2 Menit)")
spark.sql("SELECT * FROM agregasi_waktu").show(truncate=False)

>>> Batch 3 dikirim ke folder streaming...

--- [HASIL] Agregasi Cabang - Setelah Batch 3 (Terakumulasi) ---
+--------------+----------------+--------------+
|nama_cabang   |total_pendapatan|jumlah_pesanan|
+--------------+----------------+--------------+
|Semarang      |110000          |1             |
|Cabang Selatan|44000           |1             |
|Cabang Utara  |45000           |2             |
|Solo          |70000           |2             |
|Yogyakarta    |72000           |1             |
|Magelang      |131000          |3             |
|Cabang Pusat  |99000           |3             |
+--------------+----------------+--------------+


[HASIL] Agregasi Berbasis Window Waktu (2 Menit)
+------------------------------------------+----------------+--------------+
|window                                    |total_pendapatan|jumlah_pesanan|
+------------------------------------------+----------------+--------------+
|{2026-11-10 12:00:00, 2026-11-10 12:02:00}|383000          |7        

**E. Sink Berbasis Berkas (Tantangan)**

In [29]:
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

In [32]:
query3 = (
    df_stream.writeStream
    .format("parquet")
    .option("path", "tugas10_output")
    .option("checkpointLocation", "tugas10_checkpoint")
    .outputMode("append")
    .start()
)

print("Streaming Query ke Format Parquet (Sink Berkas) berhasil dimulai!")

Streaming Query ke Format Parquet (Sink Berkas) berhasil dimulai!


In [35]:
data_batch_file = (
    "pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan\n"
    "P007,Cabang Timur,Matcha,2,24000,2026-11-10 10:06:00\n"
)
with open(f"{folder_streaming}/batch_file.csv", "w") as f:
    f.write(data_batch_file)

print("Batch file baru dikirim ke folder streaming...")
time.sleep(6)

Batch file baru dikirim ke folder streaming...


In [36]:
print("\n[PEMBUKTIAN] Daftar Berkas di folder tugas10_output")
!ls -lh tugas10_output


[PEMBUKTIAN] Daftar Berkas di folder tugas10_output
total 20K
-rw-r--r-- 1 kiaa kiaa 2.3K Nov 10 12:21 part-00000-1a6b0bb8-1006-4bb1-8637-5a35443fc0a8-c000.snappy.parquet
-rw-r--r-- 1 kiaa kiaa 2.1K Nov 10 12:23 part-00000-95f16c04-c999-420a-9e6f-d3268cc4b109-c000.snappy.parquet
-rw-r--r-- 1 kiaa kiaa 2.2K Nov 10 12:21 part-00001-2e6aa066-42d0-4240-b192-eb52a8ccb6dc-c000.snappy.parquet
-rw-r--r-- 1 kiaa kiaa 2.0K Nov 10 12:21 part-00002-fb8b92d7-30b9-4309-8045-0dde169da397-c000.snappy.parquet
drwxr-xr-x 2 kiaa kiaa 4.0K Nov 10 12:23 _spark_metadata


In [37]:
print("\n[HASIL] Membaca Kembali Data dari Folder Parquet (Batch DataFrame)")
df_from_parquet = spark.read.parquet("tugas10_output")
df_from_parquet.show(truncate=False)


[HASIL] Membaca Kembali Data dari Folder Parquet (Batch DataFrame)


+----------+--------------+-----------+---+------------+-------------------+-----------+
|pesanan_id|nama_cabang   |nama_menu  |qty|harga_satuan|waktu_pesan        |total_harga|
+----------+--------------+-----------+---+------------+-------------------+-----------+
|P001      |Magelang      |Kopi Susu  |2  |18000       |2026-11-10 12:00:15|36000      |
|P002      |Solo          |Americano  |1  |20000       |2026-11-10 12:00:15|20000      |
|P003      |Magelang      |Latte      |3  |25000       |2026-11-10 12:00:15|75000      |
|P004      |Yogyakarta    |Kopi Susu  |4  |18000       |2026-11-10 12:00:15|72000      |
|P005      |Semarang      |Croissant  |5  |22000       |2026-11-10 12:01:15|110000     |
|P006      |Solo          |Latte      |2  |25000       |2026-11-10 12:01:15|50000      |
|P007      |Magelang      |Americano  |1  |20000       |2026-11-10 12:01:15|20000      |
|P005      |Cabang Utara  |Manual Brew|1  |25000       |2026-11-10 10:04:15|25000      |
|P006      |Cabang Pu

26/11/10 15:23:00 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE

**EKSPLORASI**

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, when
import shutil
import os
import time

#Hentikan session lama jika ada & siapkan folder
try:
    spark.stop()
except:
    pass

folder_streaming = "tugas10_stream"
shutil.rmtree(folder_streaming, ignore_errors=True)
os.makedirs(folder_streaming, exist_ok=True)

# Buat SparkSession yang stabil dan ringan
spark = SparkSession.builder \
    .appName("EksplorasiDiskon") \
    .master("local[1]") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Definisikan skema dan readStream
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

df_stream = (
    spark.readStream
    .schema(skema_pesanan)
    .csv(folder_streaming)
)

# Terapkan eksplorasi kolom diskon
df_diskon = df_stream.withColumn(
    "status_diskon",
    when(col("qty") >= 2, "Dapat Diskon").otherwise("Harga Normal")
)

# Jalankan query streaming ke memory sink
query_diskon = (
    df_diskon.writeStream
    .format("memory")
    .queryName("streaming_diskon")
    .outputMode("append")
    .start()
)

print("Berhasil! Streaming query diskon sudah aktif.")

#  Kirim data uji coba ke folder streaming
data_uji = (
    "pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan\n"
    "P101,Cabang Pusat,Kopi Susu,1,18000,2026-11-10 10:00:00\n"
    "P102,Cabang Utara,Espresso,3,20000,2026-11-10 10:01:00\n"
)
with open(f"{folder_streaming}/data_diskon.csv", "w") as f:
    f.write(data_uji)

time.sleep(5) # Tunggu proses stream

#  Tampilkan hasil
print("\nHASIL] Tabel streaming_diskon ")
spark.sql("SELECT * FROM streaming_diskon").show(truncate=False)

Berhasil! Streaming query diskon sudah aktif.

HASIL] Tabel streaming_diskon 
+----------+------------+---------+----+------------+-------------------+-------------+
|pesanan_id|nama_cabang |nama_menu|qty |harga_satuan|waktu_pesan        |status_diskon|
+----------+------------+---------+----+------------+-------------------+-------------+
|pesanan_id|nama_cabang |nama_menu|NULL|NULL        |NULL               |Harga Normal |
|P101      |Cabang Pusat|Kopi Susu|1   |18000       |2026-11-10 10:00:00|Harga Normal |
|P102      |Cabang Utara|Espresso |3   |20000       |2026-11-10 10:01:00|Dapat Diskon |
+----------+------------+---------+----+------------+-------------------+-------------+



In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col
import shutil
import os

# Hentikan sisa session lama
try:
    spark.stop()
except:
    pass

# Buat SparkSession paling minimalis agar tidak membebani RAM 5GB Anda
spark = SparkSession.builder \
    .appName("Tugas10_SafeBatch") \
    .master("local[1]") \
    .config("spark.driver.memory", "512m") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Buat folder dan file CSV uji coba secara langsung
folder_streaming = "tugas10_stream"
shutil.rmtree(folder_streaming, ignore_errors=True)
os.makedirs(folder_streaming, exist_ok=True)

# Tulis data dummy CSV
data_batch1 = (
    "pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan\n"
    "P001,Cabang Pusat,Kopi Susu,2,18000,2026-11-10 10:00:00\n"
    "P002,Cabang Utara,Espresso,1,20000,2026-11-10 10:01:00\n"
    "P003,Cabang Pusat,Americano,3,15000,2026-11-10 10:02:30\n"
)
with open(f"{folder_streaming}/data_uji.csv", "w") as f:
    f.write(data_batch1)

# Definisikan skema
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# Baca sebagai Batch DataFrame biasa (100% aman dari error Connection Refused / Crash RAM)
df_batch = spark.read.schema(skema_pesanan).csv(folder_streaming)

# Tambahkan kolom total harga dan tampilkan
df_hasil = df_batch.withColumn("total_harga", col("qty") * col("harga_satuan"))
print(">>> Berhasil membaca data secara stabil tanpa streaming crash!")
df_hasil.show()

26/11/10 15:40:41 WARN Utils: Your hostname, cindi resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/10 15:40:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 15:40:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


>>> Berhasil membaca data secara stabil tanpa streaming crash!


+----------+------------+---------+----+------------+-------------------+-----------+
|pesanan_id| nama_cabang|nama_menu| qty|harga_satuan|        waktu_pesan|total_harga|
+----------+------------+---------+----+------------+-------------------+-----------+
|pesanan_id| nama_cabang|nama_menu|NULL|        NULL|               NULL|       NULL|
|      P001|Cabang Pusat|Kopi Susu|   2|       18000|2026-11-10 10:00:00|      36000|
|      P002|Cabang Utara| Espresso|   1|       20000|2026-11-10 10:01:00|      20000|
|      P003|Cabang Pusat|Americano|   3|       15000|2026-11-10 10:02:30|      45000|
+----------+------------+---------+----+------------+-------------------+-----------+

